# <center>Laboratorio 9: Benchmark de Carga y Modelos con Spotify 🎵</center>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

---

### Cuerpo Docente

- Profesores: Pablo Badilla y Diego Cortez
- Auxiliares: Valentina Rojas y Melanie Peña
- Ayudantes: Javiera Arévalo, Tamara Carrasco e Ignacio Reyes

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Javier Cruz Araneda
- Nombre de alumno 2: Enzo Toledo Venegas

---

### Reglas

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibido copiar.
- Uso de LLM (Copilot, Claude, Cursor, etc.) restringido a consultas, documentación y corrección de errores.

# Temas a tratar

- Lectura eficiente de datos en formato Parquet.
- Optimización del uso de memoria mediante conversión de tipos de datos.
- Paralelización de operaciones I/O con `ThreadPoolExecutor`.
- Comparación de implementaciones de predicción: Python, NumPy, Numba, pandas y Polars.
- Entrenamiento de modelos con RandomForestRegressor y efecto de `n_jobs`.
- Orquestación de pipelines de datos con Apache Airflow.

# Objetivos principales del laboratorio

- Cargar datos de canciones de Spotify desde archivos Parquet y optimizar su representación en memoria.
- Comparar el tiempo de lectura de archivos en serie vs. en paralelo.
- Analizar el impacto de distintas implementaciones (Python puro, NumPy, Numba, pandas, Polars) en el tiempo de predicción de un modelo lineal.
- Entrenar un RandomForestRegressor que prediga la valencia de canciones, comparando el efecto de la paralelización del entrenamiento.
- Orquestar el pipeline completo (carga + entrenamiento) usando Apache Airflow.

> Instalamos e importamos las librerías necesarias 🎸

In [1]:
!uv pip install pandas pyarrow lightgbm scikit-learn plotly apache-airflow polars numba

Using Python 3.14.3 environment at: C:\Users\Javier\Desktop\MDS7202\.venv
Resolved 140 packages in 1.52s
 Downloaded grpcio
 Downloaded cryptography
 Downloaded libcst
 Downloaded polars-runtime-32
 Downloaded apache-airflow-core
Prepared 76 packages in 3.16s
Uninstalled 2 packages in 32ms
Installed 82 packages in 748ms
 + a2wsgi==1.10.10
 + aiosmtplib==5.1.2
 + aiosqlite==0.21.0
 + apache-airflow==3.2.2
 + apache-airflow-core==3.2.2
 + apache-airflow-providers-common-compat==1.15.0
 + apache-airflow-providers-common-io==1.7.3
 + apache-airflow-providers-common-sql==2.0.1
 + apache-airflow-providers-smtp==3.0.1
 + apache-airflow-providers-standard==1.14.0
 + apache-airflow-task-sdk==1.2.2
 + argcomplete==3.6.3
 + asgiref==3.11.1
 + cachetools==7.1.4
 + cadwyn==7.1.0
 + cron-descriptor==2.1.0
 + croniter==6.2.2
 + cryptography==49.0.0
 + deprecated==1.3.1
 + dill==0.4.1
 + dnspython==2.8.0
 + email-validator==2.3.0
 - fastapi==0.136.3
 + fastapi==0.138.0
 + fastapi-cli==0.0.27
 + fsspec

In [2]:
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from pathlib import Path

import numba
import numpy as np
import pandas as pd
import plotly.express as px
import polars as pl
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

DATA_DIR = Path("data")

# 1. Carga y Optimización de Datos con Parquet

Los datos que usaremos en este laboratorio corresponden a un dataset de canciones de Spotify almacenado en **20 archivos Parquet** (`batch_01.parquet` … `batch_20.parquet`), con un total de 200 000 canciones y 24 columnas que incluyen características de audio, metadatos y la letra completa de cada canción.

A continuación trabajaremos en dos aspectos fundamentales de la carga de datos en la práctica:
1. **Optimizar el uso de memoria** ajustando los tipos de datos de las columnas.
2. **Reducir el tiempo de carga** paralelizando la lectura de archivos.

## 1.1 Exploración y Optimización de Tipos de Datos [1 Punto]

Cuando cargamos datos con pandas, los tipos inferidos por defecto no siempre son los más eficientes. Por ejemplo, un entero que siempre cabe en 16 bits se almacena por defecto como `int64` (64 bits), usando 4 veces más memoria de la necesaria. Lo mismo ocurre con flotantes y con columnas categóricas almacenadas como strings.

**Código dado** — funciones de carga:

In [3]:
def load_batch(path: str) -> pd.DataFrame:
    """Lee un único archivo Parquet y retorna un DataFrame."""
    return pd.read_parquet(path)


def load_all_serial(data_dir: Path, n_batches: int | None = None) -> pd.DataFrame:
    """Lee todos los archivos Parquet de data_dir en serie y los concatena."""
    paths = sorted(data_dir.glob("*.parquet"))
    if n_batches is not None:
        paths = paths[:n_batches]
    return pd.concat([load_batch(str(p)) for p in paths], ignore_index=True)

**TO-DO [0.3 Puntos]:**
- [ ] Ejecutar `load_all_serial` sobre todos los batches y explorar el DataFrame resultante (`.dtypes`, `.memory_usage(deep=True)`).
- [ ] Aplicar las siguientes conversiones a un nuevo DataFrame (copia del originalmente cargado `df_opt`):
  - `float64` → `float32`: columnas de audio features (`danceability`, `energy`, `loudness`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, `tempo`, `avg_artist_popularity`).
  - `int64` → `int16`: columnas `key`, `mode`.
  - `int64` → `int32`: columnas `year`, `popularity`, `duration_ms`, `total_artist_followers`.
- [ ] Comparar el uso de memoria antes y después con un gráfico de barras usando Plotly (código dado).

In [4]:
# Escribe aquí tu código
df = load_all_serial(DATA_DIR, n_batches=20)

df_opt = df.copy()  # .astype(...)

print(df.dtypes)
print(f"\nMemoria total (deep): {df.memory_usage(deep=True).sum() / 1024**2:.1f} MiB")

id                            str
name                          str
album_name                    str
artists                    object
danceability              float64
energy                    float64
key                         int64
loudness                  float64
mode                        int64
speechiness               float64
acousticness              float64
instrumentalness          float64
liveness                  float64
valence                   float64
tempo                     float64
duration_ms                 int64
lyrics                        str
year                        int64
genre                         str
popularity                  int64
total_artist_followers      int64
avg_artist_popularity     float64
artist_ids                 object
niche_genres               object
dtype: object

Memoria total (deep): 358.7 MiB


In [5]:
# Aplicar las conversiones pedidas:
to_float32 = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "avg_artist_popularity",
]
to_int16 = ["key", "mode"]
to_int32 = ["year", "popularity", "duration_ms", "total_artist_followers"]

df_opt[to_float32] = df_opt[to_float32].astype("float32")
df_opt[to_int16] = df_opt[to_int16].astype("int16")
df_opt[to_int32] = df_opt[to_int32].astype("int32")

In [6]:
# Verificamos conversiones con dtype:
print(df_opt.dtypes)

id                            str
name                          str
album_name                    str
artists                    object
danceability              float32
energy                    float32
key                         int16
loudness                  float32
mode                        int16
speechiness               float32
acousticness              float32
instrumentalness          float32
liveness                  float32
valence                   float32
tempo                     float32
duration_ms                 int32
lyrics                        str
year                        int32
genre                         str
popularity                  int32
total_artist_followers      int32
avg_artist_popularity     float32
artist_ids                 object
niche_genres               object
dtype: object


In [7]:
# Compara el uso de memoria antes y después con un gráfico de barras
mem_before = df.memory_usage(deep=True).sum() / 1024**2
mem_after = df_opt.memory_usage(deep=True).sum() / 1024**2

px.bar(
    x=["Antes", "Después"],
    y=[mem_before, mem_after],
    labels={"x": "Estado", "y": "Uso de Memoria (MiB)"},
    title=f"Uso de memoria: {mem_before:.1f} MiB → {mem_after:.1f} MiB ({(1 - mem_after / mem_before) * 100:.1f}% reducción)",
).show()

In [8]:
# Veamos el promedio de largo de los strings:
string_cols = df.select_dtypes(include="object").columns
for col in string_cols:
    avg_len = df[col].str.len().mean()
    print(f"{col}: longitud promedio = {avg_len:.1f}")

id: longitud promedio = 22.0
name: longitud promedio = 18.9
album_name: longitud promedio = 20.1
artists: longitud promedio = 1.2
lyrics: longitud promedio = 1279.3
genre: longitud promedio = 5.2
artist_ids: longitud promedio = 1.2
niche_genres: longitud promedio = 2.9


C:\Users\Javier\AppData\Local\Temp\ipykernel_4432\167964972.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_cols = df.select_dtypes(include="object").columns


In [9]:
valence16 = df["valence"].astype("float16")

error = (df["valence"] - valence16.astype("float64")).abs()

print(error.max())
print(error.mean())

0.00024218750000004619
7.952466441039911e-05


### Preguntas [0.7 Puntos]

1. ¿Qué es el formato **Parquet**? ¿Qué ventajas tiene sobre CSV para datos analíticos? ¿Qué es *columnar storage* y por qué acelera las consultas que solo leen algunas columnas?
2. ¿Qué es **Apache Arrow**? ¿Cómo se relaciona con Parquet y con pandas internamente? ¿Qué ganas al usar `pd.read_parquet` en vez de `pd.read_csv`?
3. ¿Por qué existe `float32` si `float64` es más preciso? ¿En qué contextos esa pérdida de precisión es irrelevante?
4. ¿Cuándo **no** conviene reducir la precisión de un tipo numérico? ¿Qué riesgos concretos existen?
5. ¿Existe alguna alternativa a pandas para trabajar con estos datos de forma más eficiente en memoria? (menciona al menos dos)
6. ¿Cuánto se redujo el uso de memoria en total (en MiB y en %)? ¿Era esperable ese resultado? ¿Por qué no se redujo tanto como podría esperarse?
7. ¿Qué pasaría si intentaras reducir `valence` a `float16`? ¿Qué riesgo existiría para el modelo entrenado en la sección 2?

**Escribe tus respuestas aquí...**

> 1. El formato **Parquet** es un formato de almacenamiento de datos de código abierto y por columnas. La ventaja principal con respecto al formato CSV (el cual almacena datos por filas) es que permite una compresión mucho mejor en disco pues los tipos de datos por columna son homogéneos y permite seleccionar un subconjunto a trabajar y no es necesario leerlas todas. El **columnar storage** guarda los valores de una misma columna de forma contigua en disco, por tanto, una consulta que pide pocas columnas lee solo esos bloques en lugar de recorrer todas las filas, lo que reduce mucho los tiempos de lectura y escritura.

> 2. **Apache Arrow** es un formato de datos en memoria pensado para procesar y transferir datos (columnares) a alta velocidad. La relación con Parquet es que este último almacena eficientemente en disco de forma columnar y Arrow es el encargado de procesarlo eficientemente en bloques contiguos en la RAM (buffers). Por otro lado, la relación con Pandas es que este último usa Arrow internamente para, por ejemplo, usar `read_parquet`. Así, lo que se gana usando `pd.read_parquet` con respecto a `pd.read_csv` es que permite un uso selectivo de columnas y el almacenamiento en buffers permite explorar grandes conjuntos de datos con "vistas" hacia ciertas zonas de la memoria (idea de zero-copy).

> 3. `float32` existe porque usa la mitad de memoria que `float64`, lo que permite, por ejemplo, disminuir el uso de memoria tal y como se vio antes. Esta pérdida de precisión (de unos 16 decimales a 8 decimales) es irrelevante, por ejemplo, si normalizamos datos mediante algún tipo de operación aritmética que termina truncando o aproximando decimales o datos que vengan directamente de mediciones con ruido, donde la precisión decimal tan exacta deja de ser realmente de ayuda.

> 4. No conviene reducir precisión cuando hay muchos cálculos intermedios involucrados, valores con rangos muy grandes, en cálculos financieros o científicos sensibles, por ejemplo, en un contexto médico donde la precisión en ciertos cálculos puede ser fundamental para entregar un buen diagnóstico, etc.

> 5. Uno de los más populares es **Polars**, el cual viene a ser casi un reemplazo a pandas, pero escrito en Rust y utiliza Arrow internamente, por ejemplo, para ejecutar consultas de datos muy grandes prepara un plan previamente y optimiza operaciones intermedias. Otra alternativa famosa es **DuckDB** el cual permite leer directamente archivos Parquet, es muy similar a un esquema de SQL y escala muy bien en tiempos de consulta con datasets grandes.

> 6. El uso de memoria pasó de 358.7 MiB a 345.7 MiB, representando un `3.6%` de reducción o equivalentemente `13.0 MiB`. No se redujo tanto como podría esperarse pues solo hicimos trabajo sobre las variables numéricas, sin embargo, tal y como puede verse en la celda anterior, el `promedio de largo lyrics es de 1279.3` por tanto los strings son los que realmente están usando más memoria.

> 7. Al reducir valence a float16, los testeos de celdas anteriores dan un error de redondeo máximo del orden de $10^{-4}$ y medio de $10^{-5}$. En este caso el error es muy pequeño porque valence está acotada en [0,1] y float16 mantiene buena precisión en ese rango, por lo que el impacto sobre el modelo sería muy bajo. Sin embargo, el riesgo general de float16 aquí es que la precisión la tenemos realmente hasta el tercer decimal y valores muy cercanos entre sí podrían caer en el mismo número, afectando la variable objetivo y empeorando así las métricas de entrenamiento/evaluación.

In [10]:
# **IMPORTANTE**: Una vez contestada la pregunta, ejecutar esta celda para liberar memoria.
df_opt = None

## 1.2 Lectura en Serie vs. Paralelo [1 Punto]

Cuando se trabaja con múltiples archivos, la lectura **en paralelo** puede reducir el tiempo total al aprovechar que la espera de I/O (disco/red) no bloquea al procesador. En Python, la clase `ThreadPoolExecutor` del módulo `concurrent.futures` permite lanzar múltiples hilos para ejecutar operaciones de forma concurrente.

**TO-DO: [0.3 Puntos]**
- [ ] Implementar `load_all_parallel` usando `ThreadPoolExecutor`.
- [ ] Medir con `%timeit` ambas versiones sobre todos los batches.
- [ ] Generar un gráfico de línea (Plotly) con los tiempos para 2, 4, 6, …, 20 archivos, con series `Serial` y `Paralelo`.

In [11]:
# Escribe aquí tu código
def load_all_parallel(data_dir: Path, n_batches: int | None = None) -> pd.DataFrame:
    # cargar Parquets ordenados:
    paths = sorted(data_dir.glob("*.parquet"))

    # limitar o no cantidad de batches a cargar:
    if n_batches is not None:
        paths = paths[:n_batches]

    # leer archivos en paralelo con ThreadPoolExecutor:
    with ThreadPoolExecutor() as executor:
        batch_dfs = list(executor.map(load_batch, [str(p) for p in paths]))

    return pd.concat(batch_dfs, ignore_index=True)  # ignore index para resetear índices al concatenar

In [12]:
# Medicion directa sobre los 20 batches:
%timeit -n 3 -r 2 load_all_serial(DATA_DIR)
%timeit -n 3 -r 2 load_all_parallel(DATA_DIR)

427 ms ± 8.89 ms per loop (mean ± std. dev. of 2 runs, 3 loops each)
203 ms ± 827 μs per loop (mean ± std. dev. of 2 runs, 3 loops each)


**Benchmark:** mide tiempos para 2, 4, 6, ..., 20 archivos y grafica


In [13]:
@dataclass
class ReadMeasurement:
    n_files: int
    time_sec: float
    version: str


measurements: list[ReadMeasurement] = []

for n in range(2, 21):
    t0 = time.perf_counter()
    load_all_serial(DATA_DIR, n_batches=n)
    measurements.append(ReadMeasurement(n, time.perf_counter() - t0, "Serial"))

    t0 = time.perf_counter()
    load_all_parallel(DATA_DIR, n_batches=n)
    measurements.append(ReadMeasurement(n, time.perf_counter() - t0, "Paralelo"))

df_times = pd.DataFrame(measurements)
px.line(
    df_times,
    x="n_files",
    y="time_sec",
    color="version",
    markers=True,
    title="Tiempo de lectura: Serial vs Paralelo",
    labels={"n_files": "Número de archivos", "time_sec": "Tiempo (s)"},
).show()

### Preguntas  [0.7 Puntos]

1. ¿Qué significa que una operación sea **I/O-bound** vs **CPU-bound**? ¿A cuál categoría pertenece la lectura de archivos desde disco?
2. ¿Qué es el **GIL** (*Global Interpreter Lock*) de CPython? ¿Por qué existe? ¿Qué problema resuelve y qué limitación introduce?
3. ¿Por qué usamos Python si tiene el GIL? ¿Qué ganamos al usarlo como lenguaje de *pegamento* entre librerías de alto rendimiento (NumPy, Arrow, PyTorch…)?
4. ¿Cuándo conviene usar `ThreadPoolExecutor` vs `ProcessPoolExecutor`? ¿Cuál usarías si la operación fuera puramente CPU-bound?
5. ¿Qué overhead introduce crear un pool de threads? ¿Qué pasaría si los archivos fueran muy pequeños (p.ej. 1 KB cada uno)?
6. ¿Se observó mejora con la lectura paralela? ¿A partir de cuántos archivos empieza a ser notable?
7. ¿Por qué el speedup obtenido **no es igual** al número de threads disponibles? ¿Qué factores lo limitan?

**Escribe tus respuestas aquí...**

> 1. Una operación **I/O-bound** pasa la mayor parte del tiempo esperando entrada/salida (disco o red), mientras una **CPU-bound** está limitada por la capacidad de cómputo del procesador (osea la cantidad de cálculos que debe hacer). Leer archivos desde disco es I/O-bound, porque la mayor parte del tiempo el programa está esperando a que el disco entregue los datos, no calculando.

> 2. El **GIL** (Global Interpreter Lock) es un mecanismo de seguridad de Python que hace que, aunque un programa tenga varios hilos (threads), solo uno de ellos pueda ejecutar código Python a la vez. Existe porque simplifica mucho el funcionamiento interno del intérprete, especialmente la gestión de memoria. Sin el GIL, dos hilos podrían modificar los mismos datos al mismo tiempo y causar errores. El problema es que, aunque se tengan varios hilos, no se pueden usar todos los núcleos del procesador para código Python puro al mismo tiempo, porque los hilos tienen que ir turnándose (no paralelo en rigor).

> 3. Usamos Python porque realmente casi nunca hace todo el trabajo completamente solo para una tarea. Python es como un **orquestador** que coordina librerías que sí están escritas en código muy eficiente (C, C++, Rust), como NumPy, PyTorch o Arrow. Esas librerías hacen los cálculos pesados por fuera del GIL, así que ahí sí se puede usar todo el poder del hardware.

> 4. `ThreadPoolExecutor` usa hilos dentro del mismo proceso y es bueno cuando el problema es I/O-bound, porque mientras un hilo espera otro puede trabajar. `ProcessPoolExecutor` usa procesos separados, cada uno con su propio intérprete de Python. Esto sí permite paralelismo real en CPU. Así, si la tarea fuera CPU-bound pura, conviene usar ProcessPoolExecutor, porque evita el límite del GIL.

> 5. Para crear un pool de threads hay que inicializar los hilos, coordinarlos y gestionar el cambio entre ellos. Si los archivos son muy pequeños (por ejemplo, 1 KB), puede pasar que el costo de coordinar los threads puede ser mayor que el tiempo de leer los archivos. En ese caso, el paralelismo deja de ser útil y puede ser incluso más lento que algo secuencial. Por eso el paralelismo solo vale la pena cuando cada tarea tiene suficiente “trabajo” como para compensar ese overhead.

> 6. Sí, se observó una mejora clara. Con los 20 batches, la medición con `%timeit` da **984ms** en serie frente a **420ms** en paralelo, una mejora en tiempo de más del doble. En el gráfico la curva del proceso paralelizado se mantiene por debajo de la secuencial en todo momento y va mejorando a medida que aumenta el número de archivos, manteniéndose en torno a un factor. Es decir, la ventaja es notable prácticamente desde el inicio, porque cada lectura de Parquet tiene suficiente "espera de I/O" (archivos relativamente grandes) como para que solapar varias compense el overhead de los hilos que comentamos en la pregunta anterior.

> 7. El speedup que se obtiene (aprox. x2.3) no llega a escalar con el número de threads por varias razones. Primero, por la **Ley de Amdahl**, siempre hay partes del proceso que no se pueden paralelizar. En este caso, por ejemplo, la unión final de todos los DataFrames con `pd.concat` se ejecuta de forma secuencial, y también hay partes de la lectura de Parquet que pasan por Python y quedan limitadas por el GIL, así que no se paralelizan del todo. A esto se suma que el disco duro o SSD tiene un **límite de velocidad**. También hay que considerar el costo de coordinar los threads que puede notarse en tareas no tan pesadas. Así, el paralelismo sí ayuda y reduce el tiempo total, pero no escala linealmente porque el cuello de botella es variado pues parte se queda en el disco, parte en el código secuencial, y parte en la coordinación de los hilos.

# 2. Predicción de Valencia

La columna `valence` de Spotify mide el **positivismo musical** de una canción: valores cercanos a 1 indican canciones alegres y eufóricas, mientras que valores cercanos a 0 corresponden a canciones tristes o melancólicas. En esta sección analizaremos distintas formas de realizar predicciones con un modelo de regresión lineal ya entrenado, y luego entrenaremos un modelo más complejo.

## 2.1 Regresión Lineal a Mano [1.5 Puntos]

Antes de entrenar un modelo completo, veremos cómo **la elección de implementación** afecta drásticamente el rendimiento de predicción. Usaremos un modelo de regresión lineal pre-entrenado cuyos coeficientes ya están dados, e implementaremos la predicción usando cinco enfoques distintos: Python puro, NumPy, Numba (JIT), pandas y Polars.

**Código dado — carga de datos y parámetros del modelo:**

In [14]:
# Carga de datos y preparación del split
df_train = load_all_serial(DATA_DIR, n_batches=20)

PARAM_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "tempo",
    "duration_ms",
    "year",
]

X = df_train[PARAM_COLS + ["key", "mode", "genre"]]
y = df_train["valence"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
# Parámetros del modelo lineal pre-entrenado (dados)
params = {
    "danceability": 0.7718203106411208,
    "energy": 0.4252134896942928,
    "loudness": -0.008319917439312445,
    "speechiness": -0.24543273088867107,
    "acousticness": 0.10440236785191129,
    "instrumentalness": -0.11203723673701874,
    "liveness": 0.023790522969698424,
    "tempo": 0.0007885690378158087,
    "duration_ms": -4.31739613602265e-07,
    "year": -0.0036043842721972985,
}
intercept = 6.948154825159983

params_vals = list(params.values())
params_arr = np.array(params_vals, dtype=np.float32)

**Código dado — las 5 implementaciones de predicción:**

Analiza cómo cada implementación aborda el mismo problema y presta atención a las diferencias en legibilidad, concisión y (como verás en el benchmark) rendimiento.

In [16]:
def linear_regression_predict(X: np.ndarray, params: list[float]) -> list[float]:
    """Predicción con loop Python puro."""
    preds = []
    for row in X:
        val = intercept
        for j, w in enumerate(params):
            val += row[j] * w
        preds.append(val)
    return preds


def linear_regression_predict_numpy(X: np.ndarray, params: list[float]) -> np.ndarray:
    """Predicción vectorizada con NumPy."""
    return (X * np.array(params)).sum(axis=1) + intercept


@numba.njit
def linear_regression_predict_numba(X: np.ndarray, params: np.ndarray) -> np.ndarray:
    """Predicción con Numba JIT (loop compilado a código máquina)."""
    n = X.shape[0]
    preds = np.empty(n)
    for i in range(n):
        val = intercept
        for j in range(len(params)):
            val += X[i, j] * params[j]
        preds[i] = val
    return preds


def linear_regression_predict_pandas(X: pd.DataFrame, params: list[float]) -> pd.Series:
    """Predicción vectorizada con pandas (dot product)."""
    return X.dot(pd.Series(params, index=X.columns)) + intercept


def linear_regression_predict_polars(X: pl.DataFrame, params: list[float]) -> pl.Series:
    """Predicción vectorizada con Polars (expresiones lazy)."""
    weights = dict(zip(X.columns, params, strict=False))
    expr = pl.lit(intercept)
    for col, w in weights.items():
        expr = expr + pl.col(col) * w
    return X.select(expr.alias("pred"))["pred"]

**Código dado — Benchmark de las 5 implementaciones:**

In [17]:
@dataclass
class TimeMeasurement:
    time_took: float
    iteration: int
    version: str


time_measurements: list[TimeMeasurement] = []

ranges = [10, 50, 100, 250, 500, 750, 1000, *range(1001, len(X_test) + 1, 1000)]

for it in ranges:
    X_np = X_test[PARAM_COLS].iloc[:it].to_numpy(dtype=np.float32)
    X_pd = X_test[PARAM_COLS].iloc[:it]
    X_pl = pl.from_pandas(X_pd)

    for name, fn, args in [
        ("Python", linear_regression_predict, (X_np, params_vals)),
        ("NumPy", linear_regression_predict_numpy, (X_np, params_vals)),
        ("Numba-JIT", linear_regression_predict_numba, (X_np, params_arr)),
        ("Pandas", linear_regression_predict_pandas, (X_pd, params_vals)),
        ("Polars", linear_regression_predict_polars, (X_pl, params_vals)),
    ]:
        t0 = time.perf_counter()
        fn(*args)
        time_measurements.append(TimeMeasurement(time.perf_counter() - t0, it, name))

df_bench = pd.DataFrame(time_measurements)

# Gráfico 1: tiempos absolutos
px.line(
    df_bench,
    x="iteration",
    y="time_took",
    color="version",
    markers=True,
    title="Tiempos de predicción según implementación",
    labels={"iteration": "Número de filas", "time_took": "Tiempo (s)"},
).show()

# Gráfico 2: tiempos absolutos (en log)
px.line(
    df_bench,
    x="iteration",
    y="time_took",
    color="version",
    markers=True,
    title="Tiempos de predicción según implementación (en escala logarítmica)",
    labels={"iteration": "Número de filas", "time_took": "Tiempo (s)"},
    log_y=True,
).show()

# Gráfico 3: speedup relativo respecto a Python puro
pivot = df_bench.pivot(index="iteration", columns="version", values="time_took")
for col in ["NumPy", "Numba-JIT", "Pandas", "Polars"]:
    pivot[col] = pivot["Python"] / pivot[col]
pivot["Python"] = 1.0

melted = pivot.reset_index().melt(
    id_vars=["iteration"],
    value_vars=["Python", "NumPy", "Numba-JIT", "Pandas", "Polars"],
    value_name="speedup",
)
px.line(
    melted,
    x="iteration",
    y="speedup",
    color="version",
    markers=True,
    title="Speedup relativo respecto a Python puro",
    labels={"iteration": "Número de filas", "speedup": "Speedup (×)"},
).show()

### Preguntas [1.5 Puntos]

  1. ¿Qué es la vectorización en NumPy? ¿Cómo puede ejecutar operaciones sobre arrays sin loops de Python explícitos?
  2. ¿Qué es JIT (Just-In-Time compilation)? ¿Qué hace el decorador @numba.njit? ¿Qué significa el modo nopython?
  3. ¿Por qué Numba es más lento en la primera ejecución? ¿Qué es el warm-up de JIT y cómo lo manejamos en el benchmark?
  4. ¿Qué es Polars y cuáles son sus principales características como librería de datos? ¿Para qué escenarios fue diseñada y por qué ha ganado popularidad como alternativa a pandas?
  5. ¿En qué se diferencia Polars de pandas a nivel de implementación (lenguaje, modelo de ejecución, manejo de memoria)?
  6. ¿Por qué pandas puede ser más lento que NumPy aun usando operaciones vectorizadas internamente?
  7. ¿Qué son las instrucciones SIMD (Single Instruction Multiple Data)? ¿Cómo contribuyen a la aceleración de NumPy y Polars?
  8. ¿Cuándo conviene usar Numba sobre NumPy? ¿Y Polars sobre pandas para operaciones numéricas?
  9. ¿Cuál implementación fue la más rápida en tu medición? ¿Era esperable ese resultado?
  10. ¿Se observa diferencia notable entre pandas y NumPy? ¿Por qué pandas puede ser más lento o más rápido?
  11. ¿A partir de cuántas filas empieza a ser evidente la ventaja de NumPy/Numba sobre Python puro?
  12. ¿Polars fue más eficiente que pandas en tu medición? Verifica la versión de pandas instalada (pd.__version__) y comenta si crees que la versión influye en el resultado.
  13. ¿Por qué Numba puede igualar o superar a NumPy para loops numéricos simples?
  14. El benchmark excluye el costo de convertir datos a NumPy/Polars (la conversión ocurre fuera del timing). ¿Cómo cambiaría el resultado si incluyeras ese costo? ¿En qué escenarios de
  producción ese costo no existiría?
  15.  Si tuvieras que realizar esta predicción sobre 100 millones de filas en un servidor de producción, ¿qué implementación elegirías y por qué? ¿Cambiaría tu respuesta si dispusieras de
  una GPU?

**Escribe tus respuestas aquí...**

> 1. La **vectorización** en NumPy es la idea de escribir operaciones sobre arrays completos en lugar de iterar elemento por elemento con loops de Python. Esto funciona porque NumPy no ejecuta el cálculo en Python, sino en código compilado en C. Así se evita el overhead del intérprete y se aprovechan optimizaciones como caché de CPU e instrucciones SIMD.

> 2. El **JIT** (Just-In-Time compilation) es una técnica donde el código se compila a lenguaje máquina justo en el momento en que se va a ejecutar, en lugar de interpretarse línea por línea. En Numba, el decorador `@numba.njit` hace exactamente eso convirtiendo la función la primera vez que se ejecuta. El modo **nopython** significa que la función debe poder compilarse completamente a código máquina. Si Numba detecta algo que depende del intérprete de Python, no "baja de nivel", sino que falla directamente. Esto es importante porque garantiza el máximo rendimiento posible.

> 3. Numba es más lento en la primera ejecución porque antes de correr el código tiene que compilarlo (analizar tipos, optimizarlo y generar código máquina). Esto es lo que se llama el **warm-up del JIT**. En benchmarks esto es importante porque si mides esa primera ejecución, estás midiendo compilación + ejecución, no solo rendimiento real, por tanto es importante tenerlo en cuenta para diferenciar correctamente. Por eso normalmente se hace una llamada previa "de calentamiento" antes de medir.

> 4. **Polars** es una librería de DataFrames desarrollada en Rust y construida sobre Apache Arrow que está diseñada para procesamiento de datos rápido y eficiente. Sigue el enfoque de datos en columnas mencionado inicialmente, optimiza las consultas que se hacen antes de ejecutarlas y hace un bajo uso de memoria. Está pensada para análisis de datos a gran escala (una máquina) y ha ganado mucha popularidad porque es más rápida que pandas y aprovecha mejor el hardware.

> 5. Por un lado, Pandas está escrito principalmente en Python y Cython usando Numpy por debajo, evaluando todo inmediatamente y en un solo thread normalmente. Polars está escrito en Rust sobre Arrow, es multi-threaded, usa ejecución lazy opcional con optimizador de consultas, y maneja memoria en formato columnar con zero-copy cuando es posible.

> 6. Aunque pandas use operaciones vectorizadas, puede ser más lento que Numpy porque tiene una estructura con muchos más elementos y cálculos previos que debe realizar, por ejemplo, para generar índices, manejar etiquetas, etc. Numpy en ese sentido es mucho más sencillo y operando sobre arrays homogéneos.

> 7. **SIMD** (Single Instruction Multiple Data) es una técnica de hardware donde una sola instrucción se aplica a varios datos al mismo tiempo. Por ejemplo, en lugar de sumar números uno por uno, la CPU puede sumar varios floats en paralelo dentro de un solo registro. Esto es clave para el rendimiento de Numpy y Polars, porque sus operaciones internas están diseñadas para trabajar sobre arrays contiguos en memoria y aprovechar estas instrucciones vectoriales.

> 8. Numba conviene sobre NumPy cuando el cálculo no se expresa bien como operaciones vectorizadas, por ejemplo, en loops con dependencias entre iteraciones, cuando hay lógica condicional pesada, o en algoritmos elemento-a-elemento que en NumPy obligarían a crear muchos arrays intermedios. Por otro lado, Polars conviene por sobre Pandas cuando los datos son grandes, hay muchas operaciones encadenadas (donde el optimizador lazy puede fusionarlas y/o reordenarlas), o cuando se quiere aprovechar todos los núcleos, ya que Pandas es esencialmente single-thread.

> 9. Numba-JIT fue la más rápida en los tamaños grandes, ya que en 39 001 filas marcó `~88 μs`, contra `~490 μs` de NumPy, `~451 μs` de Polars y `~819 μs` de Pandas. Eso es unas 530 veces más rápido que Python puro, que tardó `~46.7 ms`. Dicho comportamiento era esperable ya que el loop está compilado a código máquina y, para un producto punto tan simple, no carga con el overhead de materializar arrays intermedios. Lo único llamativo es que en la primera llamada (10 filas) tardó `~0.5 s`, pero eso es el warm-up del JIT (compilación), no cálculo.

> 10. Sí, hay una diferencia clara pues NumPy fue más rápido que Pandas en todos los tamaños grandes. El motivo de este comportamiento es el del punto 6, ya que Pandas trabaja _por encima_ de NumPy pero agrega manejo de índices, alineación de etiquetas y validación de tipos, lo que mete un overhead constante. Pandas solo le ganaría a NumPy en operaciones donde sus rutinas especializadas (groupby, strings/categóricas) están muy optimizadas, pero para un producto punto puro NumPy gana.

> 11. El cruce ocurre casi al inicio. En 10 filas, Python puro fue de hecho el más rápido (`28.2 μs`) frente a NumPy (`37.9 μs`), porque ahí domina el overhead de armar el array. Pero ya en 50 filas NumPy (`16.4 μs`) y Numba (`7.6 μs`) le ganan claramente a Python (`66.7 μs`), y de ahí en adelante la brecha solo crece. Osea, la ventaja se vuelve evidente entre las 10 y 50 filas, y pasados unos cientos ya es de uno o dos órdenes de magnitud.

In [ ]:
print("Versión de Pandas: " + pd.__version__)

Versión de Pandas: 3.0.1


> 12.  Sí, Polars fue más eficiente que Pandas en los tamaños grandes (en 39.001 filas, `~451 μs` vs `~819 μs`), aunque en tamaños chicos van casi empatados por el overhead de inicialización de Polars. La versión instalada es Pandas 3.0.1, bastante optimizada (trae Copy-on-Write por defecto y strings respaldados por PyArrow). La versión sí debería influir, en particular, con un Pandas antiguo la brecha con Polars sería mayor. Igual, en este benchmark esas mejoras casi no aplican porque trabajamos solo con columnas numéricas, así que Pandas sigue cargando su overhead de índices y Polars se le adelanta.

> 13. Porque NumPy, aunque vectorizado, materializa arrays intermedios, por ejemplo, en `(X * params).sum(axis=1)` primero crea el array del producto y luego lo reduce, lo que hacer recorrer la memoria dos veces. Numba compila el loop a un solo paso que multiplica y acumula sobre la marcha (fusión de operaciones), sin arrays intermedios y aprovechando registros, caché y SIMD. Para una operación tan simple como esta ese ahorro de pasadas hace que Numba no solo iguale sino que supere a NumPy, como se ve en la medición (`~88 μs` vs `~490 μs` en 39.001 filas).

> 14. Si se incluyera el costo de convertir datos a NumPy/Polars, las implementaciones que parten desde Pandas cargarían ese overhead, así, en tamaños chicos podría dominar y achicar (o incluso invertir) su ventaja, mientras que en tamaños grandes la conversión se amortiza y el ranking se mantendría. Ese costo no existiría si los datos ya nacen en el formato de cómputo, por ejemplo, leyendo directo a NumPy/Arrow desde Parquet, o con un pipeline completamente escrito en Polars. En producción normalmente se convierte una sola vez y se predice muchas veces, así que es un costo fijo amortizable.

> 15. 

### 2.2 Entrenamiento y Comparación de `n_jobs` [0.5 Puntos]

Ahora entrenaremos un modelo más complejo: un **RandomForestRegressor** que usa las características de audio más una codificación del género musical para predecir `valence`. Compararemos el efecto de paralelizar el entrenamiento con el parámetro `n_jobs`.

**Código dado — pipeline encapsulado** (no modificar):

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


def build_pipeline(n_jobs: int = 1) -> Pipeline:
    # En producción este pipeline usaría LGBMRegressor; aquí usamos RandomForest
    # para ilustrar el efecto de n_jobs de forma más pronunciada.
    return Pipeline(
        [
            (
                "column_transformer",
                ColumnTransformer(
                    [
                        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["key", "mode", "genre"]),
                        (
                            "numerical",
                            "passthrough",
                            PARAM_COLS,
                        ),
                    ]
                ),
            ),
            ("random_forest", RandomForestRegressor(n_jobs=n_jobs, random_state=42)),
        ]
    )

In [ ]:
# Entrena con n_jobs=1 y mide el tiempo
pipeline_1 = build_pipeline(n_jobs=1)
t0 = time.perf_counter()
pipeline_1.fit(X_train, y_train)
time_1job = time.perf_counter() - t0

# Entrena con n_jobs=-1 y mide el tiempo
pipeline_all = build_pipeline(n_jobs=-1)
t0 = time.perf_counter()
pipeline_all.fit(X_train, y_train)
time_all_jobs = time.perf_counter() - t0

# Calcula RMSE de ambos modelos
rmse_1 = root_mean_squared_error(y_test, pipeline_1.predict(X_test))
rmse_all = root_mean_squared_error(y_test, pipeline_all.predict(X_test))

print(f"n_jobs=1  → tiempo: {time_1job:.1f}s | RMSE: {rmse_1:.4f}")
print(f"n_jobs=-1 → tiempo: {time_all_jobs:.1f}s | RMSE: {rmse_all:.4f}")

In [ ]:
# Gráficos de tiempos y RMSE
df_perf = pd.DataFrame(
    {
        "configuracion": ["n_jobs=1", "n_jobs=-1"],
        "tiempo_s": [time_1job, time_all_jobs],
        "rmse": [rmse_1, rmse_all],
    }
)

px.bar(
    df_perf,
    x="configuracion",
    y="tiempo_s",
    title="Tiempo de entrenamiento según n_jobs",
    labels={"tiempo_s": "Tiempo (s)", "configuracion": "Configuración"},
    text_auto=".1f",
).show()

px.bar(
    df_perf,
    x="configuracion",
    y="rmse",
    title="RMSE según n_jobs",
    labels={"rmse": "RMSE", "configuracion": "Configuración"},
    text_auto=".4f",
).show()

### Preguntas [0.5 Puntos]

1. ¿Qué hace el parámetro `n_jobs` en RandomForest (y en general en scikit-learn)?
2. **¿Por qué aquí sí funciona el paralelismo real sin el problema del GIL?** (Pista: RandomForest en scikit-learn usa joblib con backend de procesos o threads nativos.)
3. ¿Cuánto mejoró el tiempo con `n_jobs=-1`? 
4. ¿Fue proporcional al número de CPUs disponibles en tu máquina? ¿Por qué no?
5. ¿Hubo diferencia en RMSE entre ambas versiones? ¿Era esperable? ¿Por qué?

**Escribe tus respuestas aquí...**

# 3. Orquestación del Pipeline con Apache Airflow

En producción, los pipelines de datos y ML rara vez se ejecutan a mano desde un notebook. Se necesita:
- **Automatización**: que el pipeline corra periódicamente (diariamente, por hora…).
- **Dependencias**: que el entrenamiento solo comience si la carga de datos terminó exitosamente.
- **Monitoreo y reintentos**: que si una tarea falla, el sistema lo registre y reintente.

**Apache Airflow** resuelve exactamente esto. Define pipelines como **DAGs** (*Directed Acyclic Graphs*), donde cada nodo es una **tarea** y las aristas definen dependencias.

| Concepto | Descripción |
|----------|-------------|
| **DAG** | Grafo Dirigido Acíclico que representa el pipeline completo |
| **Operator** | Unidad de trabajo (`PythonOperator`, `BashOperator`, …) |
| **Task** | Instancia de un Operator dentro de un DAG |
| **XCom** | Mecanismo para pasar datos pequeños entre tareas |
| **schedule** | Expresión cron que indica cuándo ejecutar el DAG |

### Setup local


En la carpeta del Lab:

```bash
export AIRFLOW_HOME=$(pwd)
airflow db migrate          # inicializa la base de datos de metadata
# Ver la contraseña. Si no se en un comienzo, ejecutar airflow standalone, parar el proceso y luego ejecutar nuevamente este comando. 
cat $AIRFLOW_HOME/simple_auth_manager_passwords.json.generated 
airflow standalone       # levanta scheduler + webserver en http://localhost:8080
```

Los DAGs deben guardarse en `./dags`.

## 3.1 Implementación del DAG

**TO-DO [0.8 Puntos]:**
- [ ] Implementar `task_load_data_fn`: cargar 5 batches en paralelo, guardar en disco como Parquet y pasar la ruta a la siguiente tarea usando XCom.
- [ ] Implementar `task_train_model_fn`: recuperar la ruta de XCom, cargar el DataFrame, preparar X e y, entrenar `build_pipeline(n_jobs=-1)` e imprimir el tiempo.
- [ ] Definir la dependencia entre tareas (`load_data >> train_model`).

El siguiente bloque es el template que debes completar en tu celda de respuesta.

In [ ]:
%%writefile ~/airflow/dags/spotify_pipeline_dag.py

from pathlib import Path
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split


DATA_DIR = Path("/RUTA/ABSOLUTA/A/Labs/Lab9_v2/data")  # AJUSTA esta ruta
OUTPUT_PATH = Path("/tmp/spotify_data.parquet")

PARAM_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "tempo",
    "duration_ms",
    "year",
]


# ── Funciones auxiliares (dadas) ─────────────────────────────────────────────


def load_batch(path: str) -> pd.DataFrame:
    return pd.read_parquet(path)


def load_all_parallel(data_dir: Path, n_batches: int = 5) -> pd.DataFrame:
    paths = sorted(data_dir.glob("*.parquet"))[:n_batches]
    with ThreadPoolExecutor(max_workers=None) as executor:
        dfs = list(executor.map(load_batch, [str(p) for p in paths]))
    return pd.concat(dfs, ignore_index=True)


def build_pipeline(n_jobs: int = -1) -> Pipeline:
    return Pipeline(
        [
            (
                "column_transformer",
                ColumnTransformer(
                    [
                        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["key", "mode", "genre"]),
                        ("numerical", "passthrough", PARAM_COLS),
                    ]
                ),
            ),
            ("random_forest", RandomForestRegressor(n_jobs=n_jobs, random_state=42)),
        ]
    )


# ── Funciones de las tareas de Airflow ───────────────────────────────────────


def task_load_data_fn(**context):
    """
    Carga 5 batches de datos en paralelo y guarda el resultado en disco.
    TODO: implementa esta función.
    - Usa load_all_parallel para cargar los datos.
    - Guarda el DataFrame resultante en OUTPUT_PATH (formato parquet).
    - Usa XCom para pasar la ruta del archivo a la siguiente tarea.
    """
    ...


def task_train_model_fn(**context):
    """
    Carga los datos desde disco y entrena el pipeline.
    TODO: implementa esta función.
    - Recupera la ruta del archivo desde XCom.
    - Lee el DataFrame desde esa ruta.
    - Prepara X e y, realiza el split 80/20.
    - Entrena build_pipeline(n_jobs=-1).
    - Imprime el tiempo de entrenamiento.
    """
    ...


# ── Definición del DAG ────────────────────────────────────────────────────────

with DAG(
    dag_id="spotify_pipeline",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    tags=["mds7202", "spotify"],
) as dag:
    load_data = PythonOperator(
        task_id="load_data",
        python_callable=task_load_data_fn,
    )

    train_model = PythonOperator(
        task_id="train_model",
        python_callable=task_train_model_fn,
    )

    # TODO: define la dependencia entre tareas (load_data debe ejecutarse antes que train_model)
    ...


In [ ]:
# Escribe aquí tu código (copia el template y completa los TODOs)


Una vez guardado el archivo, ejecuta el DAG con:

```bash
airflow standalone
```


### Pega aquí el output de las steps del DAG

- Step 1: 
...


- Step 2:
...

### Preguntas [1.2 Puntos]

1. ¿Qué es un **DAG**? ¿Qué significa que sea *Directed* (dirigido) y *Acyclic* (acíclico)? ¿Por qué importa la propiedad acíclica en un pipeline de datos?
2. ¿Qué es **Apache Airflow**? ¿Para qué tipo de problemas está diseñado y cuál es su unidad mínima de trabajo?
3. ¿Qué son los **Operators**? ¿Qué diferencia hay entre `PythonOperator` y `BashOperator`? ¿Cuándo usarías cada uno?
4. ¿Qué es **XCom** en Airflow? ¿Cómo funciona internamente (¿dónde se almacena?)? ¿Por qué **no** es adecuado para pasar DataFrames grandes entre tareas?
5. ¿Qué alternativa concreta usaste para pasar el DataFrame entre `load_data` y `train_model`? ¿Cuál sería la alternativa recomendada en producción (S3, GCS, DVC…)?
6. ¿Qué es el parámetro `schedule` de un DAG? ¿Cómo lo configurarías para que corra todos los días a las 3 AM?
7. ¿Qué diferencia hay entre Airflow y otras herramientas como **Prefect**, **Dagster**, **Luigi**, **Kubeflow**? ¿Cuál es la principal crítica que se le hace a Airflow?
8. ¿Por qué conviene orquestar el pipeline en Airflow en vez de simplemente ejecutar un script Python end-to-end?
9. ¿Qué pasa si `load_data` falla a mitad de camino? ¿Airflow reintenta automáticamente? ¿Cómo controlarías el número máximo de reintentos?
10. ¿Qué ventaja tiene que las tareas estén separadas (carga y entrenamiento) vs. una sola tarea monolítica, desde el punto de vista de debugging y eficiencia?
11. ¿Cómo podemos alertar si es que algún paso falla? ¿O si la pipeline se ejecuta correctamente?
12. En un pipeline de producción real, ¿qué otras tareas añadirías al DAG?

**Escribe tus respuestas aquí...**

# Conclusión

Eso ha sido todo para el lab de hoy. Recuerda que el laboratorio tiene un plazo de entrega de una semana. Cualquier duda, no dudes en contactarnos por el foro de U-Cursos.

<p align="center">
  <img src="https://media.giphy.com/media/l0HlBO7eyXzSZkJri/giphy.gif" width="300">
</p>